In [1]:
import os
import kagglehub
import shutil

path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
print("Fichiers téléchargés dans :", path)

# Dossier de destination
dest = "data"
#os.makedirs(dest, exist_ok=True)

# Copier tous les fichiers du dossier téléchargé vers /data
#for fichier in os.listdir(path):
#    src_file = os.path.join(path, fichier)
#    dst_file = os.path.join(dest, fichier)
#    shutil.copy2(src_file, dst_file)

print("Fichiers copiés dans :", dest)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fichiers téléchargés dans : /home/onyxia/.cache/kagglehub/competitions/house-prices-advanced-regression-techniques
Fichiers copiés dans : data


In [1]:
import pandas as pd
import matplotlib as plt
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import importlib
import preprocessing as p
importlib.reload(p)

df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')

In [2]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
X_train,X_valid,y_train,y_valid=train_test_split(X,y,test_size=0.2,random_state=42)

In [3]:
test_ids = df_test['Id'].copy()
X_train,power_transform,imputer=p.preprocessing(X_train)
X_valid,_,_=p.preprocessing(X_valid,power_transform,imputer)
df_test,_,_=p.preprocessing(df_test,power_transform,imputer)

In [4]:
model = LinearRegression()
model.fit(X_train, np.log1p(y_train))

y_pred = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(np.log1p(y_valid), y_pred))
print(rmse)

0.12457570576748891


In [5]:
print("Utilities max :", X_train['Utilities'].max())      # doit être 4, pas 10^40
print("coef max abs  :", np.abs(model.coef_).max())        # doit être petit (< ~5)
print("écart moyen   :", (model.predict(X_valid) - np.log1p(y_valid).values).mean())
print("RMSE          :", np.sqrt(mean_squared_error(np.log1p(y_valid), model.predict(X_valid))))

Utilities max : 4.0
coef max abs  : 1.7975317381607656
écart moyen   : 0.0014422674554603035
RMSE          : 0.12457570576748891


In [6]:
import pandas as pd
coefs = pd.Series(model.coef_, index=X_train.columns)
top = coefs.abs().sort_values(ascending=False).head(8).index
for c in top:
    dt = X_train[c].mean()
    dv = X_valid[c].mean()
    print(f"{c:20s} coef={coefs[c]:+.2f}  train={dt:.3f}  valid={dv:.3f}  Δ={dv-dt:+.3f}")

RoofMatl_ClyTile     coef=-1.80  train=0.001  valid=0.000  Δ=-0.001
Condition2_PosN      coef=-0.66  train=0.002  valid=0.000  Δ=-0.002
3SsnPorch            coef=+0.56  train=0.002  valid=0.001  Δ=-0.000
RoofMatl_WdShngl     coef=+0.40  train=0.003  valid=0.007  Δ=+0.003
RoofMatl_WdShake     coef=+0.30  train=0.003  valid=0.007  Δ=+0.004
RoofMatl_Roll        coef=+0.30  train=0.001  valid=0.000  Δ=-0.001
Condition2_PosA      coef=+0.30  train=0.001  valid=0.000  Δ=-0.001
RoofMatl_CompShg     coef=+0.29  train=0.984  valid=0.976  Δ=-0.008


In [7]:
erreurs = np.abs(y_pred - np.log1p(y_valid).values)
idx = np.argsort(erreurs)[-5:]   # les 5 pires
print("pred :", y_pred[idx])
print("vrai :", np.log1p(y_valid).values[idx])
print("écart:", erreurs[idx])

print(X_valid.describe().loc[['min','max']].T.sort_values('max').tail())
print(X_valid.describe().loc[['min','max']].T.sort_values('min').head())

print(X_train['Utilities'].max(), X_train['Functional'].max())

pred : [11.79674359 11.69892741 11.52629067 12.40817642 11.21974844]
vrai : [11.40168101 11.28352488 11.07443601 11.8706069  10.59665973]
écart: [0.39506259 0.41540253 0.45185466 0.53756951 0.6230887 ]


                      min     max
YearBuilt     1880.000000  2009.0
GarageYrBlt   1902.570644  2009.0
YearRemodAdd  1950.000000  2010.0
YrSold        2006.000000  2010.0
BsmtUnfSF        0.000000  2042.0
              min       max
BsmtQual      0.0  5.000000
BsmtExposure  0.0  4.000000
BsmtFinType1  0.0  6.000000
MasVnrArea    0.0  3.706729
FullBath      0.0  3.000000
4.0 8.0


In [8]:
model.fit(X_train, np.log1p(y_train))
print("intercept :", model.intercept_)
print("moyenne y :", np.log1p(y_train).mean())

y_pred_train = model.predict(X_train)
print("pred train :", y_pred_train[:3])
print("vrai  train:", np.log1p(y_train).values[:3])

intercept : 7.4696108308484614
moyenne y : 12.030658310971573
pred train : [11.84577268 12.07935727 11.43166771]
vrai  train: [11.88449592 12.08954445 11.3504183 ]


In [9]:
for col in ['GrLivArea', 'LotArea', '1stFlrSF']:
    print(col, "| train:", round(X_train[col].mean(),2), round(X_train[col].std(),2),
                "| valid:", round(X_valid[col].mean(),2), round(X_valid[col].std(),2))

GrLivArea | train: 7.09 0.31 | valid: 7.05 0.33
LotArea | train: 9.24 0.53 | valid: 9.18 0.53
1stFlrSF | train: 6.41 0.26 | valid: 6.38 0.26


In [10]:
print("Utilities max:", X_train['Utilities'].max(), X_valid['Utilities'].max())
print("colonnes:", X_train.shape[1], X_valid.shape[1])
print("même ordre:", (X_train.columns == X_valid.columns).all())


Utilities max: 4.0 4.0
colonnes: 228 228
même ordre: True


In [11]:
print(X_valid.index[:5])
print(y_valid.index[:5])
print((X_valid.index == y_valid.index).all())

Index([892, 1105, 413, 522, 1036], dtype='int64')
Index([892, 1105, 413, 522, 1036], dtype='int64')
True


In [12]:
y_pred = model.predict(X_valid)
print("pred valid :", y_pred[:5])
print("vrai valid :", np.log1p(y_valid).values[:5])
print("écart moyen:", (y_pred - np.log1p(y_valid).values).mean())
print("écart std  :", (y_pred - np.log1p(y_valid).values).std())

pred valid : [11.94125496 12.73305775 11.56834503 12.04728313 12.61700215]
vrai valid : [11.94795585 12.69158354 11.6526961  11.97666577 12.66191713]
écart moyen: 0.0014422674554603035
écart std  : 0.12456735660699753


In [13]:
print("coef max abs :", np.abs(model.coef_).max())
print("nb coef > 10 :", (np.abs(model.coef_) > 10).sum())

coef max abs : 1.7975317381607656
nb coef > 10 : 0
